# Qwen2.5-VL CORD LoRA training

This notebook uses `trainer.sft_qlora_invoice_extraction`, the same trainer as the working application.
Select a compatible GPU runtime with enough free memory for the existing trainer
settings. Run `python -m scripts.prepare_colab_dataset` locally and upload
`colab_training_bundle.zip`. It includes 161 training and 20 validation receipts;
held-out test files and original Optum invoices are excluded.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

uploaded = files.upload()
assert "colab_training_bundle.zip" in uploaded
workspace = Path("/content/optum_train")
workspace.mkdir(parents=True, exist_ok=True)
with ZipFile("colab_training_bundle.zip") as archive:
    assert not any(name.startswith(("data/assignment_dataset/", "data/cord_sft_qlora/test/"))
                   for name in archive.namelist())
    assert all(not Path(name).is_absolute() and ".." not in Path(name).parts
               for name in archive.namelist())
    archive.extractall(workspace)
%cd /content/optum_train


## Prepare a separate training environment

The trainer requires a virtual environment named `.venv`. These commands apply
to the temporary Colab runtime. Review the existing pinned dependencies before
installing; preserve the environment used for the interview demo.


In [ ]:
!python -m venv --system-site-packages .venv
!.venv/bin/python -m pip install -r requirements.txt
!.venv/bin/python -c "import torch; print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')"


## Validate the selected train and validation splits

This CPU check uses manifest hashes, paired labels and image fingerprints. It
does not open held-out test files.


In [ ]:
!.venv/bin/python -c "from trainer.sft_qlora_invoice_extraction.data_prep import load_training_data; records, audit = load_training_data('data/cord_sft_qlora'); print({name: len(rows) for name, rows in records.items()}); print(audit)"
!.venv/bin/python -m trainer.sft_qlora_invoice_extraction.train --preflight


## Run an isolated one-step smoke test

Smoke output is isolated and never updates the successful adapter pointer.


In [ ]:
!.venv/bin/python -u -m trainer.sft_qlora_invoice_extraction.train --dataset-root data/cord_sft_qlora --max-steps 1 --smoke


## Train and verify the CORD adapter

Run full training only after the smoke test succeeds. Training uses validation
loss and completion-only masks; no held-out test data is used.


In [ ]:
!.venv/bin/python -u -m trainer.sft_qlora_invoice_extraction.train --dataset-root data/cord_sft_qlora --epochs 2
!.venv/bin/python -m trainer.sft_qlora_invoice_extraction.train --verify-latest


## Save the verified adapter and run metadata

Save the selected run and its pointer together. Do not copy the pretrained model
cache. Evaluation and the live application demo remain separate steps.


In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil

drive.mount("/content/drive")
root = Path("models/qwen25_finetune_sf_qlora")
pointer = json.loads((root / "latest_successful.json").read_text())
run = root / Path(pointer["adapter_path"]).parent
destination = Path("/content/drive/MyDrive/optum_cord_lora")
shutil.copytree(run, destination / run.relative_to(root), dirs_exist_ok=True)
destination.mkdir(parents=True, exist_ok=True)
shutil.copy2(root / "latest_successful.json", destination / "latest_successful.json")
print("Saved:", destination)
